# Flipkart Challenge

## Importing Libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

## Importing Train and Test Set

In [2]:
training_set = pd.read_csv('train.csv')
testing_set = pd.read_csv('test.csv')
training_set.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


## Preprocessing

### Finding Missing Value

In [3]:
percentage_missing = (training_set.isnull().mean() * 100).sort_values(ascending=False)
print(percentage_missing)

Temperature      3.227726
Weather          1.031061
RoadType         0.776207
day              0.000000
geohash          0.000000
Index            0.000000
timestamp        0.000000
NumberofLanes    0.000000
demand           0.000000
Landmarks        0.000000
LargeVehicles    0.000000
dtype: float64


### Dealing with Missing Value

In [4]:
temp_median = training_set['Temperature'].median()
training_set['Temperature'] = training_set['Temperature'].fillna(temp_median)
testing_set['Temperature'] = testing_set['Temperature'].fillna(temp_median)
training_set['Weather'] = training_set['Weather'].fillna("Unknown")
testing_set['Weather'] = testing_set['Weather'].fillna("Unknown")
training_set["RoadType"] = training_set["RoadType"].fillna("Unknown")
testing_set["RoadType"] = testing_set["RoadType"].fillna("Unknown")
training_set.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather
0,0,qp02z1,48,0:0,0.048804,Unknown,1,Not Allowed,No,16.382587,Unknown
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,16.382587,Rainy
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy


### Dealing with Geo Location

In [5]:
import pygeohash as pgh

training_set[['latitude', 'longitude']] = training_set['geohash'].apply(
    lambda x: pd.Series(pgh.decode(x))
)

testing_set[['latitude', 'longitude']] = testing_set['geohash'].apply(
    lambda x: pd.Series(pgh.decode(x))
)

In [6]:
training_set['lat_lon'] = (
    training_set['latitude'] * training_set['longitude']
)

testing_set['lat_lon'] = (
    testing_set['latitude'] * testing_set['longitude']
)

In [7]:
# Compute center from TRAIN only
lat_center = training_set['latitude'].mean()
lon_center = training_set['longitude'].mean()

training_set['dist_center'] = np.sqrt(
    (training_set['latitude'] - lat_center)**2 +
    (training_set['longitude'] - lon_center)**2
)
testing_set['dist_center'] = np.sqrt(
    (testing_set['latitude'] - lat_center)**2 +
    (testing_set['longitude'] - lon_center)**2
)

In [8]:
geohash_col = training_set['geohash'].copy()

training_set.drop(columns=['geohash'], inplace=True)
testing_set.drop(columns=['geohash'], inplace=True)

### Encoding

In [9]:
from sklearn.preprocessing import LabelEncoder

weather_le = LabelEncoder()
road_le = LabelEncoder()
vehicle_le = LabelEncoder()
landmark_le = LabelEncoder()
# longitude_le = LabelEncoder()

training_set['Weather'] = weather_le.fit_transform(training_set['Weather'])
testing_set['Weather'] = weather_le.transform(testing_set['Weather'])

training_set['RoadType'] = road_le.fit_transform(training_set['RoadType'])
testing_set['RoadType'] = road_le.transform(testing_set['RoadType'])

training_set['LargeVehicles'] = vehicle_le.fit_transform(training_set['LargeVehicles'])
testing_set['LargeVehicles'] = vehicle_le.transform(testing_set['LargeVehicles'])

training_set['Landmarks'] = landmark_le.fit_transform(training_set['Landmarks'])
testing_set['Landmarks'] = landmark_le.transform(testing_set['Landmarks'])

training_set.head()

,Index,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,latitude,longitude,lat_lon,dist_center
0,0,48,0:0,0.048804,3,1,1,0,16.382587,4,-5.484924,90.664673,-497.288869,0.168925
1,1,48,0:0,0.118507,1,3,0,1,31.104565,3,-5.462952,90.686646,-495.416761,0.138313
2,2,48,0:0,0.027132,1,1,1,0,25.919267,3,-5.462952,90.708618,-495.536796,0.127315
3,3,48,0:0,0.003272,1,1,1,0,16.382587,1,-5.462952,90.862427,-496.377045,0.150985
4,4,48,0:0,0.010819,1,1,1,0,10.803667,1,-5.457458,90.675659,-494.858647,0.140444


### Time

In [10]:
training_set['timestamp'] = pd.to_datetime(training_set['timestamp'])
testing_set['timestamp'] = pd.to_datetime(testing_set['timestamp'])

C:\Users\Suryansh\AppData\Local\Temp\ipykernel_26052\448404288.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  training_set['timestamp'] = pd.to_datetime(training_set['timestamp'])
C:\Users\Suryansh\AppData\Local\Temp\ipykernel_26052\448404288.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  testing_set['timestamp'] = pd.to_datetime(testing_set['timestamp'])


In [11]:
training_set['hour'] = training_set['timestamp'].dt.hour
testing_set['hour'] = testing_set['timestamp'].dt.hour

training_set['minute'] = training_set['timestamp'].dt.minute
testing_set['minute'] = testing_set['timestamp'].dt.minute

In [12]:
# Single feature capturing all 96 time slots of the day
training_set['time_slot'] = training_set['hour'] * 4 + training_set['minute'] // 15
testing_set['time_slot']  = testing_set['hour'] * 4 + testing_set['minute'] // 15

In [13]:
training_set.drop(columns=['hour'], inplace=True)
testing_set.drop(columns=['hour'], inplace=True)
training_set.drop(columns=['minute'], inplace=True)
testing_set.drop(columns=['minute'], inplace=True)

In [14]:
training_set.drop(columns=['timestamp'], inplace=True)
testing_set.drop(columns=['timestamp'], inplace=True)

In [15]:
training_set.head()

,Index,day,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,latitude,longitude,lat_lon,dist_center,time_slot
0,0,48,0.048804,3,1,1,0,16.382587,4,-5.484924,90.664673,-497.288869,0.168925,0
1,1,48,0.118507,1,3,0,1,31.104565,3,-5.462952,90.686646,-495.416761,0.138313,0
2,2,48,0.027132,1,1,1,0,25.919267,3,-5.462952,90.708618,-495.536796,0.127315,0
3,3,48,0.003272,1,1,1,0,16.382587,1,-5.462952,90.862427,-496.377045,0.150985,0
4,4,48,0.010819,1,1,1,0,10.803667,1,-5.457458,90.675659,-494.858647,0.140444,0


## Evaluation Function

In [16]:
from sklearn.metrics import r2_score, root_mean_squared_error, mean_absolute_error
import numpy as np

def evaluate(label, y_true, y_pred):
    rmse = root_mean_squared_error(y_true, y_pred)
    mae  = mean_absolute_error(y_true, y_pred)
    r2   = r2_score(y_true, y_pred)
    print(f"{label:<25}  RMSE: {rmse:.4f}  MAE: {mae:.4f}  R²: {r2:.4f}")

## Model Training

In [17]:
from sklearn.model_selection import train_test_split
X = training_set.drop(columns=['Index', 'demand'])
y = training_set['demand']
X_train, X_val, y_train, y_val = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

### XGBoost

In [18]:
import xgboost as xgb
xgBoost = xgb.XGBRegressor(
    objective='reg:squarederror',
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_weight=5,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method='hist',
    n_jobs=-1,
    eval_metric='rmse',
    random_state=42
)

### LightGBM

In [19]:
import lightgbm as lgb

lightGBM = lgb.LGBMRegressor(
    objective='regression',       
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=8,
    min_child_samples=5,          
    subsample=0.8,                
    colsample_bytree=0.8,         
    subsample_freq=1,             
    n_jobs=-1,
    verbose=-1,
    metric='rmse',                
    random_state=42
)

### CatBoost

In [20]:
from catboost import CatBoostRegressor

catBoost = CatBoostRegressor(
    loss_function='RMSE',         
    iterations=1000,              
    learning_rate=0.05,
    depth=8,                      
    min_data_in_leaf=5,           
    subsample=0.8,                
    colsample_bylevel=0.8,        
    bootstrap_type='Bernoulli',   
    thread_count=-1,              
    random_seed=42,
    verbose=0
)

### OOF Predictions

In [21]:
from sklearn.model_selection import KFold
from xgboost.callback import EarlyStopping

kf = KFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

oof_xgb = np.zeros(len(X_train))
oof_lgb = np.zeros(len(X_train))
oof_cat = np.zeros(len(X_train))

for train_idx, val_idx in kf.split(X_train):

    X_tr = X_train.iloc[train_idx]
    y_tr = y_train.iloc[train_idx]

    X_val_fold = X_train.iloc[val_idx]

    xgBoost.fit(X_tr, y_tr)
    lightGBM.fit(X_tr, y_tr)
    catBoost.fit(X_tr, y_tr)

    oof_xgb[val_idx] = xgBoost.predict(X_val_fold)
    oof_lgb[val_idx] = lightGBM.predict(X_val_fold)
    oof_cat[val_idx] = catBoost.predict(X_val_fold)

### Base Model Accuracy

In [22]:
evaluate("OOF XGBoost", y_train, oof_xgb)
evaluate("OOF LightGBM", y_train, oof_lgb)
evaluate("OOF CatBoost", y_train, oof_cat)

oof_avg = (oof_xgb + oof_lgb + oof_cat) / 3
evaluate("OOF Simple Average", y_train, oof_avg)

OOF XGBoost                RMSE: 0.0336  MAE: 0.0223  R²: 0.9441
OOF LightGBM               RMSE: 0.0370  MAE: 0.0253  R²: 0.9324
OOF CatBoost               RMSE: 0.0403  MAE: 0.0280  R²: 0.9197
OOF Simple Average         RMSE: 0.0360  MAE: 0.0245  R²: 0.9358


### Retrain Base Model on Full Dataset

In [23]:
xgBoost.fit(X_train, y_train)
lightGBM.fit(X_train, y_train)
catBoost.fit(X_train, y_train)

CatBoostRegressor(bootstrap_type='Bernoulli', colsample_bylevel=0.8, depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', min_data_in_leaf=5, random_seed=42, subsample=0.8, verbose=0)

### Creating Meta Features

In [24]:
val_meta = np.column_stack([
    xgBoost.predict(X_val),     
    lightGBM.predict(X_val),    
    catBoost.predict(X_val) 
])

### Train Meta Model

In [25]:
from sklearn.linear_model import Ridge

meta_model = Ridge(alpha=1.0)
meta_model.fit(val_meta, y_val)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

### Stack Score Accuracy

In [26]:
val_stack_pred = meta_model.predict(val_meta)
evaluate("Stack Val (Ridge)",  y_val, val_stack_pred)

Stack Val (Ridge)          RMSE: 0.0312  MAE: 0.0214  R²: 0.9518


### Meta Features for Test Data

In [27]:
X_test = testing_set.drop(columns=['Index'])

preds = np.column_stack([
    xgBoost.predict(X_test),
    lightGBM.predict(X_test),
    catBoost.predict(X_test)
])

### Uncertainty

In [28]:
mean_pred = preds.mean(axis=1)
std_pred = preds.std(axis=1)

# Keeping the lowest 20% std predictions as more reliable
threshold = np.percentile(
    std_pred,
    20
)

mask = std_pred <= threshold

X_pseudo = X_test[mask]
y_pseudo = mean_pred[mask]

### Retaining the New Data

In [29]:
X_new = pd.concat(
    [X_train, X_pseudo],
    ignore_index=True
)

y_new = np.concatenate(
    [y_train, y_pseudo]
)

xgBoost.fit(X_new, y_new)
lightGBM.fit(X_new, y_new)
catBoost.fit(X_new, y_new)

CatBoostRegressor(bootstrap_type='Bernoulli', colsample_bylevel=0.8, depth=8, iterations=1000, learning_rate=0.05, loss_function='RMSE', min_data_in_leaf=5, random_seed=42, subsample=0.8, verbose=0)

### Creating & Training Meta Features

In [30]:
val_meta_v2 = np.column_stack([
    xgBoost.predict(X_val),    
    lightGBM.predict(X_val),
    catBoost.predict(X_val)
])

meta_model.fit(val_meta_v2, y_val)

,"alpha alpha: float or array-like of shape (n_targets,), default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.See :ref:`sphx_glr_auto_examples_linear_model_plot_ridge_coeffs.py`for an illustration of the effect of alpha on the model coefficients.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' 

### Stack Score Accuracy 

In [31]:
evaluate("Stack Val after pseudo-label", y_val, meta_model.predict(val_meta_v2))

Stack Val after pseudo-label  RMSE: 0.0313  MAE: 0.0214  R²: 0.9515


## Prediction

### Meta Prediction

In [32]:
test_meta_v2 = np.column_stack([
    xgBoost.predict(X_test),    
    lightGBM.predict(X_test),
    catBoost.predict(X_test)
])

### Final Prediction

In [33]:
final_pred = meta_model.predict(test_meta_v2) 